# IOAI — 2024 On Site Round Lost In Hyperspace — ⭐모범답안 (Colab 자동 설정판)

이 노트북은 IOAI 로컬 연습 사이트에서 **데이터·학습환경이 자동 준비**되도록 생성되었습니다.
아래 **설정 셀을 먼저 실행**하면 공식 GitHub 저장소에서 이 문제 폴더만 부분 클론으로 받아
(전체 6.6GB 가 아니라 해당 폴더만), 그 폴더로 이동한 뒤 이후 셀이 그대로 학습/예측을 합니다.
완료 후 생성되는 제출 파일을 내려받아 연습 사이트의 **Submissions** 탭에 올리면 채점됩니다.

> 런타임 메뉴 → **런타임 유형 변경 → GPU** 로 바꾸면 학습이 빨라집니다.

In [ ]:
# === 데이터 + 환경 자동 설정 (가장 먼저 실행) ===
# 공식 공개 저장소에서 이 문제 폴더만 부분 클론(sparse)으로 받고 그 폴더로 이동한다.
import os
REPO_URL = "https://github.com/IOAI-official/IOAI-2024"
CLONE = "IOAI-2024"
SUBDIR = "On-Site-Round/Lost_in_Hyperspace"
WORKDIR = "On-Site-Round/Lost_in_Hyperspace/Solution"
# Colab 은 /content 가 홈. 재실행해도 경로가 안정적이도록 고정 기준에서 시작한다.
BASE = "/content" if os.path.isdir("/content") else os.getcwd()
os.chdir(BASE)
if not os.path.isdir(os.path.join(CLONE, SUBDIR)):
    !git clone --filter=blob:none --no-checkout --depth 1 $REPO_URL $CLONE
    !cd $CLONE && git sparse-checkout set "$SUBDIR"
    !cd $CLONE && git checkout
os.chdir(os.path.join(BASE, CLONE, WORKDIR))
print("작업 폴더:", os.getcwd())
print("내용:", sorted(os.listdir(".")))

# Lost in Hyperspace — 모범답안2 (타깃별 feature 엔지니어링)

**출처/기반**: IOAI 공식 *Best solutions* 아카이브의 ML 풀이(대칭증강 + PCA + 배경 feature, total≈1.65)와 동일
접근에서 출발해 **개선**한 버전.

**제약**(원문): 고정 모델은 **LinearRegression**, feature 만 설계. 샘플당 **≤300 feature**, val 로 학습 금지,
지도학습/사전학습 feature 추출기·AutoML 금지. 단, **'모델마다 다른 feature 집합 사용 가능'** 이 명시됨.

**개선 포인트**:
1. **축 순열 대칭(6개)** 만 증강에 사용 — 데이터는 축 순열엔 불변이나 *반사엔 불변이 아니라* 반사 증강은
   라벨을 깨뜨려 오히려 악화(검증: 반사 포함 48대칭 → 1.92).
2. **채널 통계 feature**(채널별 mean/std/min/max/sum/range + 채널쌍 차이)를 추가 → **Property0 의 RMSE 를
   크게 낮춤**(scaled 2.04 → 1.64).
3. **타깃별 feature 집합 선택**(규칙 허용): Property0 은 PCA+bg+채널통계, Property1·2 는 PCA+bg.
   선택은 **누수 없는(원본 train 분할 후 fold 내 증강) 5-fold CV** 로 — val 을 학습/선택에 쓰지 않음.

**검증셋 점수**: total_score **≈1.52** (모범답안 1.65 대비 약 8% 개선, *낮을수록 좋음*).


## 1. 데이터 로딩


In [ ]:
import os, glob, itertools, numpy as np, pandas as pd
from sklearn.linear_model import LinearRegression
from sklearn.decomposition import PCA
from sklearn.metrics import mean_squared_error

def _find(*c):
    for x in c:
        if x and os.path.exists(x): return x
    for x in c:
        g = glob.glob(x) if x else []
        if g: return g[0]
    raise FileNotFoundError(c)

DATA = _find("./training_set/ml_data_onsite_start.pickle",
             "../training_set/ml_data_onsite_start.pickle", "**/ml_data_onsite_start.pickle")
data = pd.read_pickle(DATA)
X_train, y_train = data["X"]["train"], data["y"]["train"]
X_val,   y_val   = data["X"]["val"],   data["y"]["val"]
SCALING_WEIGHTS = [100/15, 100/8, 100/100]
print("train", X_train.shape, " val", X_val.shape)


## 2. 대칭 증강(축 순열 6) · feature 함수
- `aug_perm`: 3개 공간축의 순열 6가지로 데이터 6배 증강(라벨 보존 대칭).
- `make_bg`: 배경 feature(채널 3,2 기반). `chan_feats`: 채널 통계 feature.


In [ ]:
ravel = lambda X: X.reshape((X.shape[0], -1))
PERMS = list(itertools.permutations((1, 2, 3)))

def aug_perm(X, y):
    XX = [np.moveaxis(X, (1, 2, 3), p) for p in PERMS]
    return np.concatenate(XX), np.vstack([y] * len(PERMS))

def make_bg(X):
    return np.array([_x[:, :, :, 3].ravel().min() - _x[:, :, :, 2].ravel().max() for _x in X])[:, None]

def chan_feats(X):
    f = []; ch = [X[:, :, :, :, c].reshape(len(X), -1) for c in range(6)]
    for c in range(6):
        f += [ch[c].mean(1), ch[c].std(1), ch[c].min(1), ch[c].max(1), ch[c].sum(1), ch[c].max(1) - ch[c].min(1)]
    for a, b in itertools.combinations(range(6), 2):
        d = ch[a] - ch[b]; f += [d.mean(1), d.max(1) - d.min(1)]
    return np.stack(f, axis=1)


## 3. PCA(299) + 배경 + 채널통계 → feature 행렬
채널통계는 train(증강) 통계로 z-정규화. (PCA 성분은 중첩이라 299 학습 후 슬라이스해 사용)


In [ ]:
X_aug, y_aug = aug_perm(X_train, y_train)
pca = PCA(n_components=299).fit(ravel(X_aug))

P_tr, P_va = pca.transform(ravel(X_aug)), pca.transform(ravel(X_val))
bg_tr, bg_va = make_bg(X_aug), make_bg(X_val)
cf_tr, cf_va = chan_feats(X_aug), chan_feats(X_val)
mu, sd = cf_tr.mean(0), cf_tr.std(0) + 1e-9
cf_tr, cf_va = (cf_tr - mu) / sd, (cf_va - mu) / sd
NCF = cf_tr.shape[1]
print("채널통계 feature 수:", NCF)

# 타깃별 feature 집합 (각 ≤300):
#   Property0 → PCA(299-NCF) + bg + 채널통계   /   Property1,2 → PCA(299) + bg
def feats_with_cf(P, bg, cf): return np.concatenate([P[:, :299 - NCF], bg, cf], axis=-1)
def feats_pca(P, bg):         return np.concatenate([P[:, :299], bg], axis=-1)

F_tr = [feats_with_cf(P_tr, bg_tr, cf_tr), feats_pca(P_tr, bg_tr), feats_pca(P_tr, bg_tr)]
F_va = [feats_with_cf(P_va, bg_va, cf_va), feats_pca(P_va, bg_va), feats_pca(P_va, bg_va)]
print("feature 차원 (P0,P1,P2):", [f.shape[1] for f in F_tr])


## 4. 타깃별 LinearRegression · 검증셋 평가


In [ ]:
preds = np.zeros((len(y_val), 3)); total = 0.0
for f in range(3):
    model = LinearRegression().fit(F_tr[f], y_aug[:, f])
    preds[:, f] = model.predict(F_va[f])
    rmse = mean_squared_error(preds[:, f], y_val[:, f]) ** 0.5
    scaled = rmse * SCALING_WEIGHTS[f]; total += scaled
    print(f"Property #{f}: raw RMSE={rmse:.6f}  scaled={scaled:.6f}")
total /= 3
print("=" * 16)
print(f"Total score = {total:.6f}   (모범답안 ≈1.65, 낮을수록 좋음)")


## 5. 채점용 제출 파일 (검증셋 예측 → `submission.csv`)


In [ ]:
pd.DataFrame({"ID": np.arange(len(preds)),
              "y1": preds[:, 0], "y2": preds[:, 1], "y3": preds[:, 2]}
            ).to_csv("submission.csv", index=False)
print("submission.csv (검증셋 예측) 저장:", len(preds), "행")


## 제출 파일 모으기
아래 셀을 실행하면 제출 파일이 **최상위(`/content`)로 복사**되어 왼쪽 파일 탐색기에 바로 보입니다.
그 파일을 내려받아 연습 사이트 **Submissions** 탭에 올리면 채점됩니다.

In [ ]:
# === 제출 파일을 /content 로 모으기 (마지막에 실행) ===
import os, glob, shutil
TARGETS = ['submission.csv']
OUT = "/content" if os.path.isdir("/content") else os.getcwd()
found = []
for name in TARGETS:
    hits = [name] if os.path.exists(name) else glob.glob(f"**/{name}", recursive=True)
    if not hits:
        print("아직 없음(해당 셀을 먼저 실행하세요):", name); continue
    dst = os.path.join(OUT, os.path.basename(hits[0]))
    if os.path.abspath(hits[0]) != os.path.abspath(dst):
        shutil.copy2(hits[0], dst)
    found.append(dst)
print("제출 파일 저장 위치(파일 탐색기 최상위):", found)